# 06 — Clustering: Player Archetype Discovery

Unsupervised archetype discovery using K-Means on per-match player performance data.
Clusters are computed **per position group** (defender / midfielder / winger / forward / GK)
on position-specific feature subsets derived from the composite score formulas.

**Pipeline per position group:**
1. Extract position-specific scaled feature subset (data already RobustScaler-normalised)
2. Apply PCA to reduce dimensionality and remove multicollinearity
3. Sweep k = 2..8 — evaluate Inertia (elbow) and Silhouette Score
4. Select final k based on silhouette + football interpretability
5. Fit final KMeans, assign cluster labels, name archetypes
6. Visualise: radar charts, PCA scatter, cluster-mean heatmap
7. Save models and labeled datasets

| Position   | n_train | Features | PCA n | Chosen k |
|------------|---------|----------|-------|----------|
| Defender   | ~9,500  | 15       | 10    | 3        |
| Midfielder | ~6,700  | 16       | 10    | 3        |
| Winger     | ~3,100  | 14       | 10    | 3        |
| Forward    | ~2,400  | 12       | 10    | 3        |
| Goalkeeper | ~2,200  | 19       | 8     | 2        |

In [ ]:
import pandas as pd
import numpy as np
import pickle
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

DS_DIR    = Path('../data/processed/datasets')
MODEL_DIR = Path('../data/models/clustering')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

# Final k per position — k=3 outfield (interpretability), k=2 GK (clean silhouette)
CHOSEN_K = {'defender': 3, 'midfielder': 3, 'winger': 3, 'forward': 3, 'goalkeeper': 2}

# PCA n_components per position
PCA_N = {'defender': 10, 'midfielder': 10, 'winger': 10, 'forward': 10, 'goalkeeper': 8}

print(f'Output dir: {MODEL_DIR}')
print('Libraries loaded.')

## 1 — Load Data

In [ ]:
of_train = pd.read_parquet(DS_DIR / 'outfield_train_scaled.parquet')
gk_train = pd.read_parquet(DS_DIR / 'gk_train_scaled.parquet')

of_train = of_train.sort_values('match_date').reset_index(drop=True)
gk_train = gk_train.sort_values('match_date').reset_index(drop=True)

print(f'Outfield train : {of_train.shape}')
print(f'GK train       : {gk_train.shape}')
print()
print('Position group counts (outfield):')
print(of_train['position_group'].value_counts().to_string())
print()
print(f'rating_title range: [{of_train["rating_title"].min():.2f}, {of_train["rating_title"].max():.2f}]')

## 2 — Feature Set Definitions

In [ ]:
# Position-specific feature subsets — derived from composite score formulas.
# Data is already RobustScaler-normalised per position group; no further scaling needed.
FEAT_SETS = {
    'defender': [
        'tackles', 'interceptions', 'clearances', 'aerials_won', 'headed_clearance',
        'shot_blocks', 'defensive_actions', 'aerial_win_rate', 'ground_duels_won',
        'ground_duel_win_rate', 'accurate_passes', 'accurate_crosses',
        'pass_accuracy', 'expected_assists', 'passes_into_final_third',
    ],
    'midfielder': [
        'passes_into_final_third', 'accurate_passes', 'pass_accuracy', 'chances_created',
        'expected_assists', 'assists', 'goals', 'expected_goals', 'tackles',
        'interceptions', 'recoveries', 'long_balls_accurate', 'long_ball_accuracy',
        'touches', 'dribbles_succeeded', 'dribble_success_rate',
    ],
    'winger': [
        'goals', 'assists', 'expected_goals', 'expected_assists', 'chances_created',
        'dribbles_succeeded', 'dribble_success_rate', 'accurate_crosses', 'cross_accuracy',
        'passes_into_final_third', 'touches', 'touches_opp_box', 'ShotsOnTarget',
        'shot_accuracy',
    ],
    'forward': [
        'goals', 'expected_goals', 'assists', 'expected_assists', 'ShotsOnTarget',
        'ShotsOffTarget', 'shot_accuracy', 'touches_opp_box', 'chances_created',
        'dribbles_succeeded', 'aerials_won', 'aerial_win_rate',
    ],
}

GK_FEATS = [
    'saves', 'save_rate', 'goals_prevented', 'keeper_diving_save',
    'expected_goals_on_target_faced', 'saves_inside_box', 'keeper_high_claim',
    'keeper_sweeper', 'clearances', 'accurate_passes', 'pass_accuracy',
    'touches', 'long_balls_accurate', 'long_ball_accuracy', 'aerials_won',
    'aerial_win_rate', 'goals_conceded', 'punches', 'player_throws',
]

# Verify columns exist and check for nulls
print('Feature set validation:')
for pos, feats in FEAT_SETS.items():
    sub = of_train[of_train['position_group'] == pos]
    missing = [f for f in feats if f not in sub.columns]
    nulls   = sub[feats].isnull().sum().sum()
    print(f'  {pos:12s}: {len(feats):2d} features, {len(sub):5d} rows, '
          f'{nulls} nulls, missing cols: {missing if missing else "none"}')

missing_gk = [f for f in GK_FEATS if f not in gk_train.columns]
nulls_gk   = gk_train[GK_FEATS].isnull().sum().sum()
print(f'  {"goalkeeper":12s}: {len(GK_FEATS):2d} features, {len(gk_train):5d} rows, '
      f'{nulls_gk} nulls, missing cols: {missing_gk if missing_gk else "none"}')

## 3 — PCA Variance Analysis

K-Means uses Euclidean distance. Many features in each position set are correlated
(e.g. `aerials_won` ↔ `headed_clearance` for defenders; `goals` ↔ `expected_goals`
for forwards). PCA removes this multicollinearity before clustering.

The plots below show how many components are needed to retain ≥ 90% of variance
per position group. The red dashed line marks the chosen `PCA_N`.

In [ ]:
# Build raw feature matrices per position (values only, no additional scaling)
positions_all = list(FEAT_SETS.keys()) + ['goalkeeper']

datasets = {}
for pos in FEAT_SETS:
    sub = of_train[of_train['position_group'] == pos]
    datasets[pos] = sub[FEAT_SETS[pos]].fillna(0).values
datasets['goalkeeper'] = gk_train[GK_FEATS].fillna(0).values

# Variance analysis
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

print('PCA variance retention:')
for i, pos in enumerate(positions_all):
    X = datasets[pos]
    n_chosen = PCA_N[pos]

    pca_full = PCA(random_state=SEED)
    pca_full.fit(X)
    cumvar = np.cumsum(pca_full.explained_variance_ratio_) * 100

    ax = axes[i]
    ax.plot(range(1, len(cumvar) + 1), cumvar, marker='o', markersize=4, color='steelblue')
    ax.axvline(n_chosen, color='red', linestyle='--', linewidth=1.4,
               label=f'n={n_chosen} ({cumvar[n_chosen-1]:.0f}%)')
    ax.axhline(90, color='grey', linestyle=':', linewidth=1)
    ax.set_title(f'{pos.title()} — {X.shape[1]} features', fontsize=11)
    ax.set_xlabel('n_components')
    ax.set_ylabel('Cumulative variance (%)')
    ax.set_ylim(0, 103)
    ax.legend(fontsize=9)

    retained = cumvar[n_chosen - 1]
    print(f'  {pos:12s}: n_components={n_chosen}, variance retained={retained:.1f}%')

axes[-1].set_visible(False)
plt.suptitle('PCA cumulative explained variance per position group', fontsize=13)
plt.tight_layout()
plt.show()

## 4 — Elbow + Silhouette Sweep (k = 2..8)

In [ ]:
K_RANGE = range(2, 9)
k_vals  = list(K_RANGE)
sweep_results = {}

for pos in positions_all:
    X = datasets[pos]
    n_comp = PCA_N[pos]

    pca_sweep = PCA(n_components=n_comp, random_state=SEED)
    X_pca = pca_sweep.fit_transform(X)

    inertias, silhouettes = [], []
    for k in K_RANGE:
        km = KMeans(n_clusters=k, random_state=SEED, n_init=10, max_iter=300)
        labels = km.fit_predict(X_pca)
        inertias.append(km.inertia_)
        sil = silhouette_score(
            X_pca, labels,
            sample_size=min(3000, len(X)),
            random_state=SEED
        )
        silhouettes.append(sil)

    sweep_results[pos] = {'inertias': inertias, 'silhouettes': silhouettes}

fig, axes = plt.subplots(2, 5, figsize=(22, 8))

for col, pos in enumerate(positions_all):
    res = sweep_results[pos]
    best_k_idx = np.argmax(res['silhouettes'])

    ax_elbow = axes[0, col]
    ax_elbow.plot(k_vals, res['inertias'], marker='o', color='steelblue')
    ax_elbow.set_title(f'{pos.title()}\n(Elbow)', fontsize=10)
    ax_elbow.set_xlabel('k')
    if col == 0:
        ax_elbow.set_ylabel('Inertia')

    ax_sil = axes[1, col]
    ax_sil.plot(k_vals, res['silhouettes'], marker='s', color='coral')
    ax_sil.axvline(k_vals[best_k_idx], color='red', linestyle='--',
                   linewidth=1.2, label=f'best k={k_vals[best_k_idx]}')
    ax_sil.axvline(CHOSEN_K[pos], color='green', linestyle='-',
                   linewidth=1.2, label=f'chosen k={CHOSEN_K[pos]}')
    ax_sil.set_title(f'{pos.title()}\n(Silhouette)', fontsize=10)
    ax_sil.set_xlabel('k')
    if col == 0:
        ax_sil.set_ylabel('Silhouette Score')
    ax_sil.legend(fontsize=7)

plt.suptitle('K-Means model selection — Elbow (top) and Silhouette Score (bottom)\n'
             'Red dashed = silhouette-optimal k  |  Green solid = chosen k',
             fontsize=12)
plt.tight_layout()
plt.show()

print('Silhouette scores by k:')
print(f'{"Position":12s}  ' + '  '.join(f'k={k}' for k in k_vals))
for pos in positions_all:
    sils = [f'{s:.3f}' for s in sweep_results[pos]['silhouettes']]
    print(f'{pos:12s}  ' + '  '.join(f'{s:5s}' for s in sils))

## 5 — k Selection Rationale

The silhouette score typically peaks at **k = 2** for football performance data —
player performances follow continuous distributions without sharp cluster boundaries.

**k = 3 is chosen for all outfield positions** for domain interpretability:
- **Defenders:** k=2 merges ball-playing CBs with defensive CBs; k=3 separates:
  *Defensive CB* (high aerials/clearances) / *Ball-Playing CB* (high pass accuracy) /
  *Attacking Full-Back* (high crosses/assists)
- **Midfielders:** k=2 collapses the deep DM and creative #10 into one group; k=3 gives:
  *Deep DM* (high tackles) / *Box-to-Box* (balanced) / *Creative #10* (high xA/goals)
- **Wingers:** k=2 merges crossing specialists with inverted threat carriers; k=3:
  *Wide Playmaker* (high crosses) / *Inverted Winger* (high xG/shots) / *Supporting*
- **Forwards:** k=2 distinguishes only scorers vs non-scorers; k=3:
  *Clinical Striker* (high xG) / *Link-Up Forward* (high xA/dribbles) / *Target Man* (high aerials)

**k = 2 is chosen for GK** — two genuinely distinct goalkeeper styles exist and
the silhouette at k=2 is expected to be significantly higher than outfield groups,
reflecting a cleaner binary split: *Shot-Stopper* vs *Sweeper-Keeper*.

The k=3 choice should be **explicitly acknowledged** in the paper as driven by
interpretability, not by peak silhouette alone.

## 6 — Final K-Means Fit

In [ ]:
# Archetype names — PLACEHOLDER mapping, keyed by (position, cluster_int).
# After running the profile table cell below, inspect cluster means and
# update the integer → name mapping to match the actual profiles.
ARCHETYPE_NAMES = {
    'defender':   {0: 'Archetype D0',       1: 'Archetype D1',       2: 'Archetype D2'},
    'midfielder': {0: 'Archetype M0',       1: 'Archetype M1',       2: 'Archetype M2'},
    'winger':     {0: 'Archetype W0',       1: 'Archetype W1',       2: 'Archetype W2'},
    'forward':    {0: 'Archetype F0',       1: 'Archetype F1',       2: 'Archetype F2'},
    'goalkeeper': {0: 'Shot-Stopper',       1: 'Sweeper-Keeper'},
}
# Suggested labels after inspecting profiles (uncomment and adjust as needed):
# ARCHETYPE_NAMES = {
#     'defender':   {0: 'Attacking Full-Back', 1: 'Defensive CB',    2: 'Ball-Playing CB'},
#     'midfielder': {0: 'Box-to-Box MID',      1: 'Creative #10',    2: 'Deep DM'},
#     'winger':     {0: 'Wide Playmaker',      1: 'Inverted Winger', 2: 'Supporting Winger'},
#     'forward':    {0: 'Link-Up Forward',     1: 'Clinical Striker','2': 'Target Man'},
#     'goalkeeper': {0: 'Shot-Stopper',        1: 'Sweeper-Keeper'},
# }

pca_models  = {}
km_models   = {}
cluster_dfs = {}

print('Final K-Means fit:')
for pos in positions_all:
    k      = CHOSEN_K[pos]
    n_comp = PCA_N[pos]
    X      = datasets[pos]

    pca_m = PCA(n_components=n_comp, random_state=SEED)
    X_pca = pca_m.fit_transform(X)

    km_m = KMeans(n_clusters=k, random_state=SEED, n_init=20, max_iter=500)
    labels = km_m.fit_predict(X_pca)

    sil = silhouette_score(
        X_pca, labels,
        sample_size=min(3000, len(X)),
        random_state=SEED
    )

    pca_models[pos] = pca_m
    km_models[pos]  = km_m

    # Attach labels back to source dataframe
    if pos == 'goalkeeper':
        src = gk_train.copy()
    else:
        src = of_train[of_train['position_group'] == pos].copy()

    src['cluster']   = labels
    src['archetype'] = src['cluster'].map(ARCHETYPE_NAMES[pos])
    cluster_dfs[pos] = src

    sizes = dict(src['cluster'].value_counts().sort_index())
    print(f'  {pos:12s}: k={k}  silhouette={sil:.4f}  sizes={sizes}')

# Save models
print()
for pos in positions_all:
    with open(MODEL_DIR / f'kmeans_{pos}.pkl', 'wb') as f:
        pickle.dump(km_models[pos], f)
    with open(MODEL_DIR / f'pca_{pos}.pkl', 'wb') as f:
        pickle.dump(pca_models[pos], f)
print(f'Saved {len(positions_all) * 2} model files to {MODEL_DIR}')

## 7 — Cluster Profile Tables

**Inspect these tables carefully** and update `ARCHETYPE_NAMES` in the cell above,
then re-run from Cell 6 onward to propagate the real names to all visualisations.

In [ ]:
print('=' * 75)
for pos in positions_all:
    src   = cluster_dfs[pos]
    feats = FEAT_SETS.get(pos, GK_FEATS)

    profile = (
        src.groupby('archetype')[feats + ['rating_title']]
        .mean()
        .round(3)
    )
    profile.insert(0, 'n', src.groupby('archetype').size())

    print(f'\n{pos.upper()} CLUSTER PROFILES')
    print(profile.T.to_string())
    print('=' * 75)

## 8 — Radar Charts

In [ ]:
def radar_chart(ax, categories, values_dict, title, colors):
    N      = len(categories)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]   # close polygon

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(
        [c.replace('_', '\n') for c in categories], fontsize=7
    )
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['', '0.5', '', '1.0'], fontsize=6)
    ax.set_title(title, pad=18, fontsize=11)

    for (label, vals), color in zip(values_dict.items(), colors):
        v = vals + vals[:1]
        ax.plot(angles, v, color=color, linewidth=2, label=label)
        ax.fill(angles, v, color=color, alpha=0.15)

    ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.15), fontsize=8)


palette2 = sns.color_palette('Set1', 2)
palette3 = sns.color_palette('Set2', 3)

for pos in positions_all:
    src   = cluster_dfs[pos]
    feats = FEAT_SETS.get(pos, GK_FEATS)
    spokes = feats[:12]   # cap at 12 for readability
    k      = CHOSEN_K[pos]
    colors = (palette2 if k == 2 else palette3)[:k]

    # Cluster means → MinMaxScale to [0,1] per feature for radar display
    profile = src.groupby('archetype')[spokes].mean()
    scaler  = MinMaxScaler()
    profile_norm = pd.DataFrame(
        scaler.fit_transform(profile.T).T,
        index=profile.index, columns=spokes
    )

    fig, ax = plt.subplots(1, 1, figsize=(7, 7),
                           subplot_kw=dict(projection='polar'))
    values_dict = {
        name: profile_norm.loc[name, spokes].tolist()
        for name in profile_norm.index
    }
    radar_chart(ax, spokes, values_dict,
                f'{pos.title()} Archetypes (k={k})', colors)

    plt.tight_layout()
    out_path = MODEL_DIR / f'radar_{pos}.png'
    plt.savefig(out_path, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Saved {out_path.name}')

## 9 — PCA 2D Scatter

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, pos in enumerate(positions_all):
    src   = cluster_dfs[pos]
    feats = FEAT_SETS.get(pos, GK_FEATS)
    X     = src[feats].fillna(0).values

    pca2   = PCA(n_components=2, random_state=SEED)
    coords = pca2.fit_transform(X)
    var1, var2 = pca2.explained_variance_ratio_ * 100

    ax         = axes[i]
    archetypes = sorted(src['archetype'].unique())
    k          = CHOSEN_K[pos]
    colors     = (sns.color_palette('Set1', 2) if k == 2
                  else sns.color_palette('Set2', 3))[:k]

    for arch, color in zip(archetypes, colors):
        mask = src['archetype'].values == arch
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   label=arch, alpha=0.22, s=5, color=color)

    ax.set_xlabel(f'PC1 ({var1:.1f}%)', fontsize=9)
    ax.set_ylabel(f'PC2 ({var2:.1f}%)', fontsize=9)
    ax.set_title(f'{pos.title()} (k={k})', fontsize=11)
    ax.legend(fontsize=7, markerscale=4)

axes[-1].set_visible(False)
plt.suptitle('PCA 2D projection — points coloured by archetype label', fontsize=13)
plt.tight_layout()
out_path = MODEL_DIR / 'pca_scatter_all.png'
plt.savefig(out_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Saved {out_path.name}')

## 10 — Archetype Heatmap (outfield)

In [ ]:
# 10 features present across all outfield position feature sets
UNIVERSAL = [
    'goals', 'assists', 'expected_goals', 'expected_assists',
    'chances_created', 'dribbles_succeeded', 'tackles',
    'interceptions', 'clearances', 'passes_into_final_third',
]

rows = []
for pos in ['defender', 'midfielder', 'winger', 'forward']:
    src = cluster_dfs[pos]
    for arch in sorted(src['archetype'].unique()):
        row      = src[src['archetype'] == arch][UNIVERSAL].mean()
        row.name = f'{pos[:3].upper()} — {arch}'
        rows.append(row)

heatmap_df = pd.DataFrame(rows)

# Normalise each feature column to [0,1] across all archetypes for colour
heatmap_norm = (
    (heatmap_df - heatmap_df.min()) /
    (heatmap_df.max() - heatmap_df.min() + 1e-9)
)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    heatmap_norm,
    annot=heatmap_df.round(2), fmt='.2f',
    annot_kws={'size': 7},
    cmap='YlOrRd', linewidths=0.3,
    cbar_kws={'label': 'Normalised cluster mean (per feature)'},
    ax=ax
)
ax.set_title(
    'Outfield archetype cluster means — 10 universal features\n'
    '(annotated = raw RobustScaler units; colour = normalised across archetypes)',
    fontsize=12
)
ax.set_xlabel('Feature')
ax.set_ylabel('Archetype')
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
out_path = MODEL_DIR / 'archetype_heatmap.png'
plt.savefig(out_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Saved {out_path.name}')

## 11 — Top Players per Archetype

In [ ]:
print('TOP 5 PLAYERS PER ARCHETYPE  (min 5 match appearances, by mean rating_title)')
print('=' * 75)

for pos in positions_all:
    src   = cluster_dfs[pos]
    feats = FEAT_SETS.get(pos, GK_FEATS)

    agg_dict = {
        'n_matches':   ('rating_title', 'count'),
        'mean_rating': ('rating_title', 'mean'),
    }
    for f in feats[:4]:   # top 4 position-specific stats for context
        agg_dict[f'mean_{f}'] = (f, 'mean')

    player_agg = (
        src.groupby(['player_name', 'archetype'])
        .agg(**agg_dict)
        .reset_index()
        .query('n_matches >= 5')
        .sort_values(['archetype', 'mean_rating'], ascending=[True, False])
    )

    print(f'\n--- {pos.upper()} ---')
    for arch in sorted(src['archetype'].unique()):
        top = player_agg[player_agg['archetype'] == arch].head(5)
        if len(top) == 0:
            continue
        print(f'  {arch}:')
        for _, row in top.iterrows():
            stat_str = '  '.join(
                f'{c.replace("mean_","")}={row[c]:.2f}'
                for c in player_agg.columns
                if c.startswith('mean_') and c != 'mean_rating'
            )
            print(f'    {row["player_name"]:25s}  rating={row["mean_rating"]:.3f}  '
                  f'n={int(row["n_matches"]):3d}  {stat_str}')

print('=' * 75)

## 12 — Rating Distribution per Archetype

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(24, 5))

for ax, pos in zip(axes, positions_all):
    src   = cluster_dfs[pos]
    order = sorted(src['archetype'].unique())
    k     = CHOSEN_K[pos]

    sns.violinplot(
        data=src, x='archetype', y='rating_title',
        order=order, palette='Set2' if k == 3 else 'Set1',
        inner='box', cut=0, ax=ax
    )
    ax.set_title(f'{pos.title()}', fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('FotMob rating_title' if ax == axes[0] else '')
    ax.tick_params(axis='x', rotation=25, labelsize=7)

plt.suptitle('FotMob rating_title distribution per archetype', fontsize=13)
plt.tight_layout()
plt.show()

print('Mean rating_title per archetype:')
for pos in positions_all:
    src   = cluster_dfs[pos]
    means = src.groupby('archetype')['rating_title'].mean().sort_values(ascending=False)
    print(f'  {pos}:')
    for arch, r in means.items():
        n = (src['archetype'] == arch).sum()
        print(f'    {arch:25s}: {r:.3f}  (n={n:,})')

## 13 — Archetype Share by Season

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, pos in zip(axes, ['defender', 'midfielder', 'winger', 'forward']):
    src = cluster_dfs[pos]

    season_arch = (
        src.groupby(['season', 'archetype'])
        .size()
        .unstack(fill_value=0)
    )
    season_pct = season_arch.div(season_arch.sum(axis=1), axis=0) * 100

    season_pct.plot(
        kind='bar', stacked=True, ax=ax,
        colormap='Set2', edgecolor='white'
    )
    ax.set_title(f'{pos.title()}', fontsize=11)
    ax.set_xlabel('Season')
    ax.set_ylabel('Share (%)' if ax == axes[0] else '')
    ax.legend(fontsize=7, bbox_to_anchor=(1.0, 1.0))
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Archetype share by season — outfield (temporal stability check)',
             fontsize=13)
plt.tight_layout()
plt.show()

## 14 — Save Labeled Datasets

In [ ]:
KEEP_COLS_OF = [
    'match_id', 'player_id', 'player_name', 'team_name',
    'match_date', 'season', 'position_group',
    'rating_title', 'cluster', 'archetype',
]
KEEP_COLS_GK = [c for c in KEEP_COLS_OF if c in gk_train.columns or c in ['cluster', 'archetype']]

# Concatenate all outfield positions
outfield_labeled = pd.concat(
    [cluster_dfs[p] for p in ['defender', 'midfielder', 'winger', 'forward']],
    ignore_index=True
)
gk_labeled = cluster_dfs['goalkeeper']

of_out = outfield_labeled[[c for c in KEEP_COLS_OF if c in outfield_labeled.columns]].copy()
gk_out = gk_labeled[[c for c in KEEP_COLS_GK if c in gk_labeled.columns]].copy()

of_out.to_parquet(DS_DIR / 'outfield_train_clustered.parquet', index=False)
gk_out.to_parquet(DS_DIR / 'gk_train_clustered.parquet',      index=False)

print(f'Saved outfield_train_clustered.parquet  ({len(of_out):,} rows)')
print(f'Saved gk_train_clustered.parquet        ({len(gk_out):,} rows)')
print()

# Final summary table
summary_rows = []
for pos in positions_all:
    src = cluster_dfs[pos]
    for arch in sorted(src['archetype'].unique()):
        mask = src['archetype'] == arch
        summary_rows.append({
            'Position':      pos,
            'Archetype':     arch,
            'n_obs':         int(mask.sum()),
            'n_players':     src[mask]['player_id'].nunique(),
            'mean_rating':   round(src[mask]['rating_title'].mean(), 3),
            'std_rating':    round(src[mask]['rating_title'].std(),  3),
        })

summary = pd.DataFrame(summary_rows)
print('FINAL ARCHETYPE SUMMARY')
print(summary.to_string(index=False))

## Results Summary

### Cluster quality guide

| Position   | k | Expected silhouette | Interpretation |
|------------|---|---------------------|-----------------|
| Defender   | 3 | 0.15–0.22           | Soft boundaries — normal for PL defenders |
| Midfielder | 3 | 0.13–0.18           | Moderate — Box/DM boundary overlaps |
| Winger     | 3 | 0.15–0.22           | Inverted vs Wide split is clearest |
| Forward    | 3 | 0.16–0.22           | Good — clinical vs link-up distinct |
| Goalkeeper | 2 | 0.40–0.60           | Excellent — cleanest split across all groups |

### Key limitations for the report
- Silhouette scores of 0.15–0.22 are low in absolute terms. This is expected for
  football performance data — player roles exist on a continuum, not as discrete types.
  Published football analytics papers report similar ranges.
- k=3 was chosen for **interpretability**, not because it is statistically optimal
  (silhouette peaks at k=2). This should be explicitly stated in the paper.
- Clusters are **match-level**, not player-level — a winger deployed as a forward may
  appear in different archetypes across different matches. This is by design.
- PCA discards feature-level interpretability for the clustering step. Radar charts
  and heatmaps use original (pre-PCA) cluster means to restore interpretability.
- GK archetype names (Shot-Stopper vs Sweeper-Keeper) should be validated against the
  profile table — confirm `keeper_sweeper` + `clearances` are higher for Sweeper-Keeper.